<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/LightBGM_20260817.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#Jensen
import datetime
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings('ignore')

# 嘗試匯入 display（相容 Jupyter 與一般 .py 腳本）
try:
  from IPython.display import display
except ImportError:

  def display(x):
    print(x)


# 1. 配置股票池
stock_dict = {

   # --- 【新增：指定台股 ETF 族群】 ---
    "0050.TW": "元大台灣50",
    "0056.TW": "元大高股息",
    "00878.TW": "國泰永續高股息",
    "00770.TW": "國泰北美科技",
    "00981A.TW": "統一台股增長主動式",
    # --- 【新增：美股重點個股與 ETF】 ---
    "AAPL": "Apple 蘋果",
    "GOOG": "Google / Alphabet",
    "META": "Meta",
    "MSFT": "Microsoft 微軟",
    "NVDA": "NVIDIA 輝達",
    "TSM": "台積電 ADR",
    "TSLA": "Tesla 特斯拉",
    "ENTG": "Entegris 英特格",
    "SMR": "NuScale Power 小型核反應爐",
    "BE": "Bloom Energy 燃料電池",
    "JNJ": "Johnson & Johnson 嬌生",
    "SPCX": "SPACs ETF",
    # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝與設備概念股】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21. 家登概念股 / 半導體載具與設備夥伴】 ---
    "2404.TW": "漢唐",
    "1773.TW": "勝一",
    # --- 【22. 光學鏡頭與先進設備】 ---
    "3008.TW": "大立光",
    "4915.TW": "先進光",
    "5288.TW": "匯鑽科",
    "3644.TWO": "凌嘉科",
    "7769.TW": "鴻勁",
    "1303.TW": "南亞",
    # --- 【23. 工業電腦 (IPC) 相關概念股】 ---
    "2395.TW": "研華",
    "6166.TW": "凌華",
    "8050.TWO": "廣積",
    "3556.TWO": "禾瑞亞",
    "2414.TW": "精技",
    "6414.TW": "樺漢",
    "3022.TW": "威強電",
    "2397.TW": "友通",
    "5314.TWO": "世紀",
    "2393.TW": "億光",
    "2465.TW": "麗臺",
    "5536.TWO": "聖暉*",
    # --- 【24. 電動車 (EV) 與車用電子相關概念股】 ---
    "2201.TW": "裕隆",
    "2204.TW": "中華",
    "2206.TW": "三陽工業",
    "1536.TW": "和大",
    "2231.TW": "聯嘉",
    "3552.TWO": "同致",
    "6279.TWO": "胡連",
    # --- 【25. 其他與基礎建設】 ---
    "2353.TW": "宏碁",
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "6613.TWO": "朋億*",
    "4755.TW": "三福化",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    "1717.TW": "長興",
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    "8358.TWO": "金居",
    "8021.TW": "尖點",
    "6672.TW": "騰輝電子-KY",
}


def apply_labeling(df, upper=0.10, lower=-0.05, horizon=10):
  closes = df['Close'].values
  highs = df['High'].values
  lows = df['Low'].values
  labels = np.zeros(len(df))
  for i in range(len(df) - horizon):
    entry_p = closes[i]
    hit = 0
    for step in range(1, horizon + 1):
      if (lows[i + step] - entry_p) / entry_p <= lower:
        break
      if (highs[i + step] - entry_p) / entry_p >= upper:
        hit = 1
        break
    labels[i] = hit
  df['Target'] = labels
  return df


def compute_enhanced_features(df, market_df):
  d = df.copy()
  # 個股特徵
  ma5 = d['Close'].rolling(5).mean()
  ma10 = d['Close'].rolling(10).mean()
  ma20 = d['Close'].rolling(20).mean()
  d['MA_Tangle'] = (
      pd.concat([ma5, ma10, ma20], axis=1).max(axis=1)
      - pd.concat([ma5, ma10, ma20], axis=1).min(axis=1)
  ) / (ma20 + 1e-6)
  d['Vol_5MA'] = d['Volume'].rolling(5).mean()
  d['Vol_Ratio'] = d['Volume'] / (d['Vol_5MA'] + 1e-6)
  d['BIAS_5'] = (d['Close'] - ma5) / (ma5 + 1e-6)

  # 大盤增強特徵
  m_close = market_df['Close']
  d['Mkt_Alpha'] = d['Close'].pct_change(5) - m_close.pct_change(5)
  delta = m_close.diff()
  gain = delta.where(delta > 0, 0).rolling(14).mean()
  loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
  rs = gain / (loss + 1e-6)
  d['Mkt_RSI'] = 100 - (100 / (1 + rs))
  d['Mkt_Vola'] = m_close.pct_change().rolling(20).std()

  cols = ['Vol_Ratio', 'BIAS_5', 'Mkt_Alpha', 'Mkt_RSI', 'Mkt_Vola']
  for c in cols:
    d[c] = d[c].shift(1)
  return d, cols


# 下載大盤數據
print('正在下載大盤 (^TWII) 數據...')
market_df = yf.download('^TWII', period='4y', progress=False)
market_df.columns = (
    market_df.columns.get_level_values(0)
    if isinstance(market_df.columns, pd.MultiIndex)
    else market_df.columns
)

all_results = []
models_cache = {}  # 儲存訓練好的模型與特徵欄位，供盤後預測直接調用
split_date = '2026-03-31'

print('\n開始訓練各個個股獨立模型 (回測階段)...')
for ticker, name in stock_dict.items():
  try:
    df = yf.download(ticker, period='4y', progress=False)
    if len(df) < 250:
      continue
    df.columns = (
        df.columns.get_level_values(0)
        if isinstance(df.columns, pd.MultiIndex)
        else df.columns
    )

    df = apply_labeling(df)
    df, f_cols = compute_enhanced_features(df, market_df)
    df = df.dropna(subset=f_cols + ['Target'])

    # 儲存完整資料供後續即時預測使用
    models_cache[ticker] = {'df_full': df, 'f_cols': f_cols}

    train = df[df.index <= split_date]
    test = df[(df.index > split_date) & (df.index <= '2026-06-30')]

    if len(train) < 100 or len(test) == 0:
      continue

    # 訓練回測模型
    model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
        verbose=-1,
    )
    model.fit(train[f_cols], train['Target'])

    test = test.copy()
    test['Prob'] = model.predict_proba(test[f_cols])[:, 1]
    test['Ticker'] = ticker
    test['Name'] = name
    all_results.append(test[['Ticker', 'Name', 'Target', 'Prob']])
  except Exception as e:
    continue

if all_results:
  final_test = pd.concat(all_results).sort_values(by='Prob', ascending=False)
  threshold = 0.51
  signals = final_test[final_test['Prob'] >= threshold]
  print(
      f'\n=== LGBM_ST2: 每檔股票獨立建模 + 大盤特徵 (門檻 >='
      f' {int(threshold*100)}%) ==='
  )
  if not signals.empty:
    precision = (signals['Target'].sum() / len(signals)) * 100
    print(
        f"總訊號數: {len(signals)} | 成功數: {int(signals['Target'].sum())} |"
        f' 勝率: {precision:.2f}%'
    )
    display(signals.head(20))
  else:
    print('無符合門檻之訊號')

# --- 最新盤後預測 (直接使用快取模型訓練至最新，省去重複下載) ---
print('\n正在產生最新交易建議...')
prediction_results = []

for ticker, name in stock_dict.items():
  if ticker not in models_cache:
    continue
  try:
    # 抓取最新 1 個月資料來取得最新特徵值
    df_latest = yf.download(ticker, period='1mo', progress=False)
    if len(df_latest) < 20:
      continue
    df_latest.columns = (
        df_latest.columns.get_level_values(0)
        if isinstance(df_latest.columns, pd.MultiIndex)
        else df_latest.columns
    )

    df_feat, f_cols = compute_enhanced_features(df_latest, market_df)
    latest_X = df_feat[f_cols].tail(1)
    if latest_X.isnull().values.any():
      continue

    # 使用該股票快取的完整歷史資料重新訓練至最新狀態
    cached_data = models_cache[ticker]
    df_train = cached_data['df_full'].dropna(
        subset=cached_data['f_cols'] + ['Target']
    )

    final_model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
        verbose=-1,
    )
    final_model.fit(df_train[cached_data['f_cols']], df_train['Target'])

    prob = final_model.predict_proba(latest_X)[0][1]

    prediction_results.append({
        '股票名稱': name,
        '股票代號': ticker.split('.')[0],
        '預測漲幅機率': f'{round(float(prob) * 100, 2)}%',
        'raw_prob': float(prob),
    })
  except Exception as e:
    continue

suggestion_df = pd.DataFrame(prediction_results).sort_values(
    by='raw_prob', ascending=False
)

print(f'\n=== LGBM_ST2: {datetime.date.today()} 盤後預測建議 (明日交易參考) ===')
print(f'篩選準則：預測未來10日漲幅達10%之機率 >= {int(threshold*100)}%')

triggered = suggestion_df[suggestion_df['raw_prob'] >= 0.51]
if not triggered.empty:
  display(triggered[['股票名稱', '股票代號', '預測漲幅機率']])
else:
  print('今日無符合門檻之強勢標的。')


正在下載大盤 (^TWII) 數據...

開始訓練各個個股獨立模型 (回測階段)...

=== LGBM_ST2: 每檔股票獨立建模 + 大盤特徵 (門檻 >= 51%) ===
總訊號數: 2347 | 成功數: 1072 | 勝率: 45.68%


Price,Ticker,Name,Target,Prob
Date,,,,
2026-06-05,2337.TW,旺宏,0.0,0.990794
2026-06-05,2303.TW,聯電,0.0,0.985666
2026-06-04,2303.TW,聯電,0.0,0.985666
2026-04-30,3131.TWO,弘塑,0.0,0.981530
2026-04-29,6187.TWO,萬潤,1.0,0.972757
2026-05-12,6187.TWO,萬潤,0.0,0.972148
2026-04-28,6187.TWO,萬潤,1.0,0.971618
2026-06-12,6147.TWO,頎邦,1.0,0.969056
2026-04-28,2342.TW,茂矽,1.0,0.947322



正在產生最新交易建議...

=== LGBM_ST2: 2026-08-17 盤後預測建議 (明日交易參考) ===
篩選準則：預測未來10日漲幅達10%之機率 >= 51%


,股票名稱,股票代號,預測漲幅機率
120,上詮,3363,52.23%


In [4]:
#無均線糾結升級為lightgbm優化模式
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import yfinance as yf

# 忽略不必要的警告訊息
warnings.filterwarnings("ignore")

# 股票代號與名稱對應字典
stock_dict = {
    # --- 【新增：指定台股 ETF 族群】 ---
    "0050.TW": "元大台灣50",
    "0056.TW": "元大高股息",
    "00878.TW": "國泰永續高股息",
    "00770.TW": "國泰北美科技",
    "00981A.TW": "統一台股增長主動式",
    # --- 【新增：美股重點個股與 ETF】 ---
    "AAPL": "Apple 蘋果",
    "GOOG": "Google / Alphabet",
    "META": "Meta",
    "MSFT": "Microsoft 微軟",
    "NVDA": "NVIDIA 輝達",
    "TSM": "台積電 ADR",
    "TSLA": "Tesla 特斯拉",
    "ENTG": "Entegris 英特格",
    "SMR": "NuScale Power 小型核反應爐",
    "BE": "Bloom Energy 燃料電池",
    "JNJ": "Johnson & Johnson 嬌生",
    "SPCX": "SPACs ETF",
    # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝與設備概念股】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21. 家登概念股 / 半導體載具與設備夥伴】 ---
    "2404.TW": "漢唐",
    "1773.TW": "勝一",
    # --- 【22. 光學鏡頭與先進設備】 ---
    "3008.TW": "大立光",
    "4915.TW": "先進光",
    "5288.TW": "匯鑽科",
    "3644.TWO": "凌嘉科",
    "7769.TW": "鴻勁",
    "1303.TW": "南亞",
    # --- 【23. 工業電腦 (IPC) 相關概念股】 ---
    "2395.TW": "研華",
    "6166.TW": "凌華",
    "8050.TWO": "廣積",
    "3556.TWO": "禾瑞亞",
    "2414.TW": "精技",
    "6414.TW": "樺漢",
    "3022.TW": "威強電",
    "2397.TW": "友通",
    "5314.TWO": "世紀",
    "2393.TW": "億光",
    "2465.TW": "麗臺",
    "5536.TWO": "聖暉*",
    # --- 【24. 電動車 (EV) 與車用電子相關概念股】 ---
    "2201.TW": "裕隆",
    "2204.TW": "中華",
    "2206.TW": "三陽工業",
    "1536.TW": "和大",
    "2231.TW": "聯嘉",
    "3552.TWO": "同致",
    "6279.TWO": "胡連",
    # --- 【25. 其他與基礎建設】 ---
    "2353.TW": "宏碁",
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "6613.TWO": "朋億*",
    "4755.TW": "三福化",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    "1717.TW": "長興",
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    "8358.TWO": "金居",
    "8021.TW": "尖點",
    "6672.TW": "騰輝電子-KY",
}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理 (LightGBM 優化版)"""
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  # 特徵位移防洩漏
  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟零：正在下載大盤基準資料 (^TWII)...")
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

print(
    "步驟一：開始進行個股批次下載與運算，並產出最近 5 個交易日的逐日獨立模型預測..."
)
predictions = []

# 批次下載所有股票資料以大幅提升速度
all_tickers = list(stock_dict.keys())
batch_data = yf.download(all_tickers, period="2y", progress=False, group_by="ticker")

for ticker, name in stock_dict.items():
  try:
    if len(all_tickers) > 1:
      df = batch_data[ticker].dropna(how="all")
    else:
      df = batch_data.dropna(how="all")

    if df.empty or len(df) < 250:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # 標籤定義：未來 10 個交易日內最高價曾漲幅達 10%
    horizon = 10
    threshold = 0.10
    future_max = (
        df["Close"]
        .shift(-1)
        .iloc[::-1]
        .rolling(window=horizon, min_periods=1)
        .max()
        .iloc[::-1]
    )
    future_return = (future_max - df["Close"]) / df["Close"]
    df["Target"] = (future_return >= threshold).astype(int)

    # 執行特徵工程與防洩漏處理
    df_feat, feature_cols = compute_features(df, market_df)

    df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
    if len(df_clean) < 60:
      continue

    # 迴圈計算最後 5 個交易日的逐日預測
    for i in range(-5, 0):
      X_train = df_clean[feature_cols].iloc[:i]
      y_train = df_clean["Target"].iloc[:i]

      if len(y_train) < 50 or len(y_train.unique()) < 2:
        continue

      # 極端值處理 (Winsorization 1% 至 99%)
      lower_bound = X_train.quantile(0.01)
      upper_bound = X_train.quantile(0.99)
      X_train_clipped = X_train.clip(lower_bound, upper_bound, axis=1)

      train_base = y_train.mean()

      # 切分出驗證集供 LightGBM 早停機制使用
      if len(X_train_clipped) > 30:
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_clipped, y_train, test_size=0.2, shuffle=False
        )
      else:
        X_tr, X_val, y_tr, y_val = (
            X_train_clipped,
            X_train_clipped,
            y_train,
            y_train,
        )

      # 建立 LightGBM 模型並加入早停機制
      model = lgb.LGBMClassifier(
          n_estimators=300,
          learning_rate=0.03,
          max_depth=4,
          random_state=42,
          verbose=-1,
      )

      model.fit(
          X_tr,
          y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
      )

      # 取得目標當天的特徵
      target_features = df_clean[feature_cols].iloc[[i]].copy()
      target_features = target_features.clip(lower_bound, upper_bound, axis=1)

      if target_features.dropna().empty:
        continue

      prob = model.predict_proba(target_features)[0][1]
      lift_val = float(prob) / float(train_base) if train_base > 0 else 0.0
      pred_date = df_clean.index[i].strftime("%Y-%m-%d")

      predictions.append({
          "預測日期": pred_date,
          "股票名稱": name,
          "股票代號": ticker.split(".")[0],
          "Base y": f"{round(float(train_base) * 100, 2)}%",
          "raw_prob": float(prob),
          "y成立機率值": f"{round(float(prob) * 100, 2)}%",
          "10% Lift 值": f"{round(lift_val, 2)}x",
      })
  except Exception:
    pass

final_output_df = pd.DataFrame(predictions)
if not final_output_df.empty:
  final_output_df = final_output_df.sort_values(
      by=["預測日期", "raw_prob"], ascending=[False, False]
  )

  print("\n" + "=" * 75)
  print(" 最近 5 個交易日個股逐日預測結果清單 (LightGBM + 早停機制優化版) ")
  print("=" * 75)
  print(
      final_output_df[[
          "預測日期",
          "股票名稱",
          "股票代號",
          "Base y",
          "y成立機率值",
          "10% Lift 值",
      ]].to_markdown(index=False)
  )
else:
  print("目前無法產生預測結果。")


步驟零：正在下載大盤基準資料 (^TWII)...
步驟一：開始進行個股批次下載與運算，並產出最近 5 個交易日的逐日獨立模型預測...

 最近 5 個交易日個股逐日預測結果清單 (LightGBM + 早停機制優化版) 
| 預測日期   | 股票名稱                   | 股票代號   | Base y   | y成立機率值   | 10% Lift 值   |
|:-----------|:---------------------------|:-----------|:---------|:--------------|:--------------|
| 2026-08-17 | 旺矽                       | 6223       | 48.93%   | 64.08%        | 1.31x         |
| 2026-08-17 | 萬潤                       | 6187       | 34.61%   | 59.41%        | 1.72x         |
| 2026-08-17 | 鈺創                       | 5351       | 36.52%   | 57.52%        | 1.58x         |
| 2026-08-17 | AES-KY                     | 6781       | 33.17%   | 57.08%        | 1.72x         |
| 2026-08-17 | 欣興                       | 3037       | 42.72%   | 56.68%        | 1.33x         |
| 2026-08-17 | 志聖                       | 2467       | 38.9%    | 56.34%        | 1.45x         |
| 2026-08-17 | 德宏                       | 5475       | 51.07%   | 55.41%        | 1.08x         |
| 2026-08-17 | 南電

In [5]:
#均線糾結並升級為Lightbgm優化模式
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import yfinance as yf

# 忽略不必要的警告訊息
warnings.filterwarnings("ignore")

# 股票代號與名稱對應字典
stock_dict = {

   # --- 【新增：指定台股 ETF 族群】 ---
    "0050.TW": "元大台灣50",
    "0056.TW": "元大高股息",
    "00878.TW": "國泰永續高股息",
    "00770.TW": "國泰北美科技",
    "00981A.TW": "統一台股增長主動式",
    # --- 【新增：美股重點個股與 ETF】 ---
    "AAPL": "Apple 蘋果",
    "GOOG": "Google / Alphabet",
    "META": "Meta",
    "MSFT": "Microsoft 微軟",
    "NVDA": "NVIDIA 輝達",
    "TSM": "台積電 ADR",
    "TSLA": "Tesla 特斯拉",
    "ENTG": "Entegris 英特格",
    "SMR": "NuScale Power 小型核反應爐",
    "BE": "Bloom Energy 燃料電池",
    "JNJ": "Johnson & Johnson 嬌生",
    "SPCX": "SPACs ETF",
    # --- 【1. AI 伺服器與 ODM 概念股】 ---
    "2382.TW": "廣達",
    "3231.TW": "緯創",
    "6669.TW": "緯穎",
    "2317.TW": "鴻海",
    "2356.TW": "英業達",
    "2324.TW": "仁寶",
    "2376.TW": "技嘉",
    "3706.TW": "神達",
    "2377.TW": "微星",
    "2357.TW": "華碩",
    "4938.TW": "和碩",
    "3005.TW": "神基",
    "5274.TWO": "信驊",
    # --- 【2. 湧德與磁性元件 / 網通高速連接器概念股】 ---
    "3689.TWO": "湧德",
    "3357.TWO": "臺慶科",
    "6862.TW": "三集瑞-KY",
    "6821.TWO": "聯寶",
    "3207.TWO": "耀勝",
    "6197.TW": "佳必琪",
    "8103.TW": "瀚荃",
    "3526.TWO": "凡甲",
    "3605.TW": "宏致",
    # --- 【3. 網通 / 網路設備概念股】 ---
    "2345.TW": "智邦",
    "5388.TW": "中磊",
    "3558.TWO": "神準",
    "3704.TW": "合勤控",
    "4906.TW": "正文",
    # --- 【4. AI伺服器機殼 / 機構件與電子零組件概念股】 ---
    "8210.TW": "勤誠",
    "6117.TW": "迎廣",
    "6235.TW": "華孚",
    "2354.TW": "鴻準",
    "3376.TW": "新日興",
    "3548.TWO": "兆利",
    "5243.TW": "乙盛-KY",
    # --- 【5. 高速傳輸 / 介面IC / PCIe / USB4 概念股】 ---
    "4966.TWO": "譜瑞-KY",
    "5269.TW": "祥碩",
    "6104.TWO": "創惟",
    "6756.TW": "威鋒電子",
    "6715.TW": "嘉基",
    # --- 【6. AI 連接器 / 高速傳輸概念股】 ---
    "3533.TW": "嘉澤",
    "3217.TWO": "優群",
    "3023.TW": "信邦",
    "2392.TW": "正崴",
    # --- 【7. AI 核心 / 晶片設計 / ASIC 概念股】 ---
    "3035.TW": "智原",
    "6643.TWO": "M31",
    # --- 【8. AI電源供應器 / HVDC / 伺服器電源概念股】 ---
    "2308.TW": "台達電",
    "2301.TW": "光寶科",
    "6282.TW": "康舒",
    "6412.TW": "群電",
    "3665.TW": "貿聯-KY",
    # --- 【9. 液冷散熱 / 散熱模組概念股】 ---
    "3017.TW": "奇鋐",
    "3324.TWO": "雙鴻",
    "3653.TW": "健策",
    "2421.TW": "建準",
    "8996.TW": "高力",
    "3483.TWO": "力致",
    "6230.TW": "尼得科超眾",
    "3013.TW": "晟銘電",
    "6805.TW": "富世達",
    # --- 【10. BBU 備援電池模組相關概念股】 ---
    "6781.TW": "AES-KY",
    "3211.TWO": "順達",
    "6121.TWO": "新普",
    "3323.TWO": "加百裕",
    "3625.TWO": "西勝",
    "8038.TWO": "長園科",
    "4931.TWO": "新盛力",
    # --- 【11. 電力概念股 (重電、變壓器、電線電纜、儲能)】 ---
    "1519.TW": "華城",
    "1513.TW": "中興電",
    "1514.TW": "亞力",
    "1503.TW": "士電",
    "1609.TW": "大亞",
    "1605.TW": "華新",
    "1608.TW": "華榮",
    "6869.TW": "雲豹能源",
    # --- 【12. PCB / ABF載板 / CCL 相關概念股】 ---
    "3037.TW": "欣興",
    "8046.TW": "南電",
    "3189.TW": "景碩",
    "4958.TW": "臻鼎-KY",
    "2368.TW": "金像電",
    "3044.TW": "健鼎",
    "2313.TW": "華通",
    "8155.TWO": "博智",
    "2383.TW": "台光電",
    "6274.TWO": "台燿",
    "6213.TW": "聯茂",
    # --- 【13. 記憶體相關概念股】 ---
    "2344.TW": "華邦電",
    "2408.TW": "南亞科",
    "2337.TW": "旺宏",
    "3006.TW": "晶豪科",
    "3260.TWO": "威剛",
    "2451.TW": "創見",
    "4967.TW": "十銓",
    "8271.TW": "宇瞻",
    "5289.TWO": "宜鼎",
    "8299.TWO": "群聯",
    "5351.TWO": "鈺創",
    # --- 【14. 光通訊 / 矽光子 / CPO 概念股】 ---
    "4979.TWO": "華星光",
    "6442.TW": "光聖",
    "4908.TWO": "前鼎",
    "3163.TWO": "波若威",
    "3450.TW": "聯鈞",
    "6426.TW": "統新",
    "4977.TW": "眾達-KY",
    "6530.TWO": "創威",
    "3363.TWO": "上詮",
    "3234.TWO": "光環",
    "4903.TWO": "聯光通",
    "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY",
    "4971.TWO": "IET-KY",
    "6588.TWO": "東典光電",
    # --- 【15. 低軌衛星 / 太空通訊概念股】 ---
    "3491.TWO": "昇達科",
    "2314.TW": "台揚",
    "6285.TW": "啟碁",
    "3105.TWO": "穩懋",
    "2455.TW": "全新",
    "3138.TW": "耀登",
    "2419.TW": "仲琦",
    # --- 【16. 被動元件族群】 ---
    "2327.TW": "國巨",
    "2492.TW": "華新科",
    "2375.TW": "凱美",
    "2478.TW": "大毅",
    "3026.TW": "禾伸堂",
    "3090.TW": "日電貿",
    "6173.TWO": "信昌電",
    "6155.TW": "鈞寶",
    "6175.TWO": "立敦",
    "5328.TWO": "華容",
    "3236.TWO": "千如",
    # --- 【17. 機器人相關概念股】 ---
    "2049.TW": "上銀",
    "4576.TW": "大銀微系統",
    "4585.TW": "達明",
    "2359.TW": "所羅門",
    "6188.TWO": "廣明",
    "8374.TW": "羅昇",
    "5443.TWO": "均豪",
    "6640.TWO": "均華",
    "2464.TW": "盟立",
    "6215.TW": "和椿",
    "4562.TW": "穎漢",
    "1590.TW": "亞德客-KY",
    "1504.TW": "東元",
    # --- 【18. 半導體封裝與設備概念股】 ---
    "3711.TW": "日月光投控",
    "2449.TW": "京元電子",
    "6257.TW": "矽格",
    "3264.TWO": "欣銓",
    "6239.TW": "力成",
    "2329.TW": "華泰",
    "2441.TW": "超豐",
    "3131.TWO": "弘塑",
    "3583.TW": "辛耘",
    "6187.TWO": "萬潤",
    "2467.TW": "志聖",
    "8027.TWO": "鈦昇",
    "3481.TW": "群創",
    "2409.TW": "友達",
    # --- 【19. 半導體應用材料 / 耗材概念股】 ---
    "5434.TW": "崇越",
    "3010.TW": "華立",
    "1560.TW": "中砂",
    "3680.TWO": "家登",
    "5234.TW": "達興材料",
    "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體",
    "6515.TW": "穎崴",
    "6683.TWO": "雍智科技",
    "6510.TWO": "精測",
    "6223.TWO": "旺矽",
    # --- 【20. 核心半導體 / IC設計 / 晶圓代工】 ---
    "2330.TW": "台積電",
    "2303.TW": "聯電",
    "2454.TW": "聯發科",
    "3034.TW": "聯詠",
    "3661.TW": "世芯-KY",
    "3443.TW": "創意",
    "4961.TW": "天鈺",
    "6415.TW": "矽力-KY",
    "6531.TW": "愛普*",
    # --- 【21. 家登概念股 / 半導體載具與設備夥伴】 ---
    "2404.TW": "漢唐",
    "1773.TW": "勝一",
    # --- 【22. 光學鏡頭與先進設備】 ---
    "3008.TW": "大立光",
    "4915.TW": "先進光",
    "5288.TW": "匯鑽科",
    "3644.TWO": "凌嘉科",
    "7769.TW": "鴻勁",
    "1303.TW": "南亞",
    # --- 【23. 工業電腦 (IPC) 相關概念股】 ---
    "2395.TW": "研華",
    "6166.TW": "凌華",
    "8050.TWO": "廣積",
    "3556.TWO": "禾瑞亞",
    "2414.TW": "精技",
    "6414.TW": "樺漢",
    "3022.TW": "威強電",
    "2397.TW": "友通",
    "5314.TWO": "世紀",
    "2393.TW": "億光",
    "2465.TW": "麗臺",
    "5536.TWO": "聖暉*",
    # --- 【24. 電動車 (EV) 與車用電子相關概念股】 ---
    "2201.TW": "裕隆",
    "2204.TW": "中華",
    "2206.TW": "三陽工業",
    "1536.TW": "和大",
    "2231.TW": "聯嘉",
    "3552.TWO": "同致",
    "6279.TWO": "胡連",
    # --- 【25. 其他與基礎建設】 ---
    "2353.TW": "宏碁",
    "8163.TW": "達方",
    "8043.TWO": "蜜望實",
    "2892.TW": "第一金",
    "5880.TW": "合庫金",
    "1210.TW": "大成",
    "1215.TW": "卜蜂",
    "1216.TW": "統一",
    "2912.TW": "統一超",
    "5903.TWO": "全家",
    "6770.TW": "力積電",
    "2342.TW": "茂矽",
    "3707.TWO": "漢磊",
    "3016.TW": "嘉晶",
    "6196.TW": "帆宣",
    "6139.TW": "亞翔",
    "6613.TWO": "朋億*",
    "4755.TW": "三福化",
    "4768.TWO": "晶呈科技",
    "3563.TW": "牧德",
    "1717.TW": "長興",
    "1815.TWO": "富喬",
    "1802.TW": "台玻",
    "5340.TWO": "建榮",
    "5475.TWO": "德宏",
    "3167.TW": "大量",
    "6438.TW": "迅得",
    "1595.TWO": "川寶",
    "6147.TWO": "頎邦",
    "8150.TW": "南茂",
    "6552.TW": "易華電",
    "3305.TW": "昇貿",
    "3631.TWO": "晟楠",
    "2059.TW": "川湖",
    "6584.TWO": "南俊國際",
    "8358.TWO": "金居",
    "8021.TW": "尖點",
    "6672.TW": "騰輝電子-KY",

}


def compute_features(df, market_df):
  """執行 4 大類特徵工程與防洩漏處理 (LightGBM 優化版)"""
  d = df.copy()

  # --- A. 價格型態與波動度特徵 ---
  d["Close_Slope"] = (
      d["Close"].rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )

  body = np.abs(d["Close"] - d["Open"])
  body_safe = np.where(body == 0, 1e-6, body)
  d["Upper_Shadow_Ratio"] = (
      d["High"] - np.maximum(d["Close"], d["Open"])
  ) / body_safe
  d["Lower_Shadow_Ratio"] = (
      np.minimum(d["Close"], d["Open"]) - d["Low"]
  ) / body_safe

  d["Gap"] = (d["Open"] - d["Close"].shift(1)) / d["Close"].shift(1)

  high_low = d["High"] - d["Low"]
  high_close = np.abs(d["High"] - d["Close"].shift(1))
  low_close = np.abs(d["Low"] - d["Close"].shift(1))
  tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
  atr14 = tr.rolling(14).mean()
  d["NATR"] = atr14 / d["Close"]

  ma20 = d["Close"].rolling(20).mean()
  std20 = d["Close"].rolling(20).std()
  upper_band = ma20 + (2 * std20)
  lower_band = ma20 - (2 * std20)
  d["BB_Bandwidth"] = (upper_band - lower_band) / ma20

  ma5 = d["Close"].rolling(5).mean()
  d["BIAS_5"] = (d["Close"] - ma5) / ma5

  # --- B. 量能與資金成本特徵 ---
  vol_mean5 = d["Volume"].rolling(5).mean()
  d["Volume_Explosion"] = d["Volume"] / (vol_mean5 + 1e-6)

  typical_price = (d["High"] + d["Low"] + d["Close"]) / 3
  vwap = (typical_price * d["Volume"]).rolling(5).sum() / (
      d["Volume"].rolling(5).sum() + 1e-6
  )
  d["VWAP_BIAS"] = (d["Close"] - vwap) / vwap

  obv = (np.sign(d["Close"].diff()) * d["Volume"]).fillna(0).cumsum()
  d["OBV_Slope"] = (
      obv.rolling(5).apply(lambda x: np.polyfit(range(5), x, 1)[0], raw=True)
  )
  d["Turnover_Rate"] = d["Volume"] / (d["Volume"].rolling(60).mean() + 1e-6)

  # --- C. 深層籌碼與信用交易特徵 ---
  d["Foreign_Buy_Ratio"] = 0.0
  d["Trust_Buy_Ratio"] = 0.0
  d["Inst_Sync"] = 0
  d["Margin_Change_5d"] = 0.0
  d["Short_Margin_Ratio"] = 0.0

  # --- D. 市場相對強度特徵 ---
  stock_ret5 = d["Close"].pct_change(5)
  market_ret5 = market_df["Close"].pct_change(5)
  d["Alpha_5d"] = stock_ret5 - market_ret5

  feature_cols = [
      "Close_Slope",
      "Upper_Shadow_Ratio",
      "Lower_Shadow_Ratio",
      "Gap",
      "NATR",
      "BB_Bandwidth",
      "BIAS_5",
      "Volume_Explosion",
      "VWAP_BIAS",
      "OBV_Slope",
      "Turnover_Rate",
      "Foreign_Buy_Ratio",
      "Trust_Buy_Ratio",
      "Inst_Sync",
      "Margin_Change_5d",
      "Short_Margin_Ratio",
      "Alpha_5d",
  ]

  # 特徵位移防洩漏
  for col in feature_cols:
    d[col] = d[col].shift(1)

  return d, feature_cols


print("步驟零：正在下載大盤基準資料 (^TWII)...")
market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex):
  market_df.columns = market_df.columns.get_level_values(0)

print(
    "步驟一：開始進行個股批次下載與運算，並套用流動性與均線糾結篩選條件..."
)
predictions = []

all_tickers = list(stock_dict.keys())
batch_data = yf.download(all_tickers, period="2y", progress=False, group_by="ticker")

for ticker, name in stock_dict.items():
  try:
    if len(all_tickers) > 1:
      df = batch_data[ticker].dropna(how="all")
    else:
      df = batch_data.dropna(how="all")

    if df.empty or len(df) < 250:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # 計算篩選條件所需的指標（直接使用當天實際數據，不作 shift）
    vol_mean_5 = df["Volume"].rolling(5).mean()
    ma5 = df["Close"].rolling(5).mean()
    ma10 = df["Close"].rolling(10).mean()
    ma20 = df["Close"].rolling(20).mean()
    ma_max = pd.concat([ma5, ma10, ma20], axis=1).max(axis=1)
    ma_min = pd.concat([ma5, ma10, ma20], axis=1).min(axis=1)

    # 條件 1：5日均量 >= 500 張 (500,000 股)
    liquidity_ok = vol_mean_5 >= 500000
    # 條件 2：5日線、10日線、20日線差距在 3% 以內
    ma_tangle = (ma_max - ma_min) / (ma_min + 1e-6) <= 0.03

    # 標籤定義：未來 10 個交易日內最高價曾漲幅達 10%
    horizon = 10
    threshold = 0.10
    future_max = (
        df["Close"]
        .shift(-1)
        .iloc[::-1]
        .rolling(window=horizon, min_periods=1)
        .max()
        .iloc[::-1]
    )
    future_return = (future_max - df["Close"]) / df["Close"]
    df["Target"] = (future_return >= threshold).astype(int)

    # 執行特徵工程與防洩漏處理
    df_feat, feature_cols = compute_features(df, market_df)

    df_clean = df_feat.dropna(subset=feature_cols + ["Target"])
    if len(df_clean) < 60:
      continue

    # 迴圈計算最後 5 個交易日的逐日預測
    for i in range(-5, 0):
      # 取得當前預測日期的 Index 標籤
      target_idx = df_clean.index[i]

      # 套用篩選條件：若當天不符合流動性或均線糾結，則略過
      if not liquidity_ok.loc[target_idx] or not ma_tangle.loc[target_idx]:
        continue

      X_train = df_clean[feature_cols].iloc[:i]
      y_train = df_clean["Target"].iloc[:i]

      if len(y_train) < 50 or len(y_train.unique()) < 2:
        continue

      # 極端值處理 (Winsorization 1% 至 99%)
      lower_bound = X_train.quantile(0.01)
      upper_bound = X_train.quantile(0.99)
      X_train_clipped = X_train.clip(lower_bound, upper_bound, axis=1)

      train_base = y_train.mean()

      # 切分出驗證集供 LightGBM 早停機制使用
      if len(X_train_clipped) > 30:
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_clipped, y_train, test_size=0.2, shuffle=False
        )
      else:
        X_tr, X_val, y_tr, y_val = (
            X_train_clipped,
            X_train_clipped,
            y_train,
            y_train,
        )

      # 建立 LightGBM 模型並加入早停機制
      model = lgb.LGBMClassifier(
          n_estimators=300,
          learning_rate=0.03,
          max_depth=4,
          random_state=42,
          verbose=-1,
      )

      model.fit(
          X_tr,
          y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
      )

      # 取得目標當天的特徵
      target_features = df_clean[feature_cols].iloc[[i]].copy()
      target_features = target_features.clip(lower_bound, upper_bound, axis=1)

      if target_features.dropna().empty:
        continue

      prob = model.predict_proba(target_features)[0][1]
      lift_val = float(prob) / float(train_base) if train_base > 0 else 0.0
      pred_date = target_idx.strftime("%Y-%m-%d")

      predictions.append({
          "預測日期": pred_date,
          "股票名稱": name,
          "股票代號": ticker.split(".")[0],
          "Base y": f"{round(float(train_base) * 100, 2)}%",
          "raw_prob": float(prob),
          "y成立機率值": f"{round(float(prob) * 100, 2)}%",
          "10% Lift 值": f"{round(lift_val, 2)}x",
      })
  except Exception:
    pass

final_output_df = pd.DataFrame(predictions)
if not final_output_df.empty:
  final_output_df = final_output_df.sort_values(
      by=["預測日期", "raw_prob"], ascending=[False, False]
  )

  print("\n" + "=" * 75)
  print(
      " 最近 5 個交易日符合篩選條件之個股逐日預測結果清單 (流動性 & 均線糾結過濾版)"
  )
  print("=" * 75)
  print(
      final_output_df[[
          "預測日期",
          "股票名稱",
          "股票代號",
          "Base y",
          "y成立機率值",
          "10% Lift 值",
      ]].to_markdown(index=False)
  )
else:
  print("目前最近 5 個交易日內，沒有符合「5日均量 >= 500張 且 均線糾結 3%」的股票。")


步驟零：正在下載大盤基準資料 (^TWII)...
步驟一：開始進行個股批次下載與運算，並套用流動性與均線糾結篩選條件...

 最近 5 個交易日符合篩選條件之個股逐日預測結果清單 (流動性 & 均線糾結過濾版)
| 預測日期   | 股票名稱          | 股票代號   | Base y   | y成立機率值   | 10% Lift 值   |
|:-----------|:------------------|:-----------|:---------|:--------------|:--------------|
| 2026-08-17 | 貿聯-KY           | 3665       | 42.0%    | 40.66%        | 0.97x         |
| 2026-08-17 | 亞翔              | 6139       | 36.75%   | 38.24%        | 1.04x         |
| 2026-08-17 | 大毅              | 2478       | 28.16%   | 38.21%        | 1.36x         |
| 2026-08-17 | 智邦              | 2345       | 36.28%   | 37.06%        | 1.02x         |
| 2026-08-17 | 矽格              | 6257       | 25.78%   | 35.97%        | 1.4x          |
| 2026-08-17 | 鈦昇              | 8027       | 29.59%   | 35.75%        | 1.21x         |
| 2026-08-17 | 威剛              | 3260       | 33.41%   | 34.42%        | 1.03x         |
| 2026-08-17 | 弘塑              | 3131       | 36.04%   | 33.88%        | 0.94x         |
| 2026-08-17 | 辛